# sweet.tv — corrected 80% / top-five target
Elizaveta Gill. Code and checks assisted by Codex. CPU only; no credentials or viewing records embedded. Main window selects configurations; forward window is opened only after freezing. Earlier 300-second/all-unique results are invalid as a Kaggle proxy and preserved in Git history.

In [ ]:
import os,sys,subprocess
from pathlib import Path
os.environ["OPENBLAS_NUM_THREADS"]="2"
os.environ["OMP_NUM_THREADS"]="2"
os.chdir("/kaggle/working")
Path("revised").mkdir(exist_ok=True)


In [ ]:
Path('core.py').write_text('"""Auditable pure-Python metric, interval join and submission contract.\n\nNo competition data or credentials are embedded. Times must be seconds on a\nsingle, verified axis. All intervals use half-open [start, end) semantics.\n"""\nfrom bisect import bisect_left, bisect_right\nfrom collections import defaultdict\nimport csv\nimport math\n\n\ndef ap_at_k(actual, predicted, k=5):\n    """Kaggle-style AP@k; empty truth is 0, duplicates get no second credit."""\n    if k < 1:\n        raise ValueError(\'k must be positive\')\n    actual = set(actual)\n    if not actual:\n        return 0.0\n    seen, hits, total = set(), 0, 0.0\n    for rank, item in enumerate(predicted[:k], 1):\n        if item in actual and item not in seen:\n            hits += 1\n            total += hits / rank\n        seen.add(item)\n    return total / min(k, len(actual))\n\n\ndef map_at_k(truth, predictions, users, k=5):\n    """Fixed evaluation cohort, INCLUDING users without relevant validation items."""\n    users = list(users)\n    if not users or len(users) != len(set(users)):\n        raise ValueError(\'Evaluation users must be unique and nonempty\')\n    if set(predictions) != set(users):\n        raise ValueError(\'Predictions must cover the exact evaluation cohort\')\n    return sum(ap_at_k(truth.get(u, ()), predictions[u], k) for u in users) / len(users)\n\n\ndef union_length(intervals):\n    """Seconds watched without double counting overlapping view-log intervals."""\n    end, total = None, 0\n    for left, right in sorted(intervals):\n        if right <= left:\n            continue\n        if end is None or left > end:\n            total += right-left\n        elif right > end:\n            total += right-end\n        end = right if end is None else max(end, right)\n    return total\n\n\nclass ScheduleIndex:\n    """Supports overlapping broadcasts, not just a single as-of match.\n\n    rows: (channel, start_seconds, end_seconds, nonzero_item_id, airing_key).\n    Prefix maximum ends allow efficient removal of all broadcasts ending before\n    a viewing starts, even when the schedule itself contains overlapping slots.\n    """\n    def __init__(self, rows):\n        by_channel = defaultdict(list)\n        for channel, start, end, item, key in rows:\n            if end <= start:\n                raise ValueError(\'Broadcast length must be positive\')\n            if item != 0:\n                by_channel[channel].append((start, end, item, key))\n        self.channels = {}\n        for channel, values in by_channel.items():\n            values.sort()\n            starts, prefix, largest = [], [], -math.inf\n            for start, end, _, _ in values:\n                starts.append(start)\n                largest=max(largest, end)\n                prefix.append(largest)\n            self.channels[channel]=(values, starts, prefix)\n\n    def overlaps(self, channel, start, end):\n        if end <= start:\n            return []\n        if channel not in self.channels:\n            return []\n        values, starts, prefix = self.channels[channel]\n        lo=bisect_right(prefix, start)\n        hi=bisect_left(starts, end)\n        result=[]\n        for bstart, bend, item, key in values[lo:hi]:\n            left, right=max(start,bstart), min(end,bend)\n            if right > left:\n                result.append((item,key,left,right,bend-bstart))\n        return result\n\n\ndef aggregate_watches(index, views, window_start, window_end, min_seconds):\n    """Clip at split BEFORE matching and merge duplicate watches per airing.\n\n    views: (user,channel,start,end). Returns qualified per-airing interactions.\n    The duration threshold is a parameter; its final value requires data audit.\n    """\n    if window_end <= window_start or min_seconds <= 0:\n        raise ValueError(\'Invalid window or threshold\')\n    segments=defaultdict(list)\n    for user,channel,start,end in views:\n        left,right=max(start,window_start),min(end,window_end)\n        for item,key,a,b,length in index.overlaps(channel,left,right):\n            segments[(user,item,key,length)].append((a,b))\n    results=[]\n    for (user,item,key,length),intervals in segments.items():\n        seconds=union_length(intervals)\n        if seconds >= min_seconds:\n            results.append({\'user_id\':user,\'tv_show_id\':item,\'airing_key\':key,\n                            \'watch_seconds\':seconds,\'programme_seconds\':length,\n                            \'watch_fraction\':seconds/length})\n    return results\n\n\ndef validate_submission(template_path, submission_path, candidates):\n    """Use the ACTUAL template\'s header/order, never an assumed column name."""\n    def read(path):\n        with open(path,newline=\'\',encoding=\'utf-8-sig\') as stream:\n            rows=list(csv.reader(stream))\n        return rows[0],rows[1:]\n    header,template=read(template_path)\n    out_header,rows=read(submission_path)\n    if out_header != header or len(header)!=2 or len(rows)!=len(template):\n        raise ValueError(\'Header or row count differs from real template\')\n    if any(len(r)!=2 for r in rows):\n        raise ValueError(\'Unexpected CSV field count\')\n    if [r[0] for r in rows] != [r[0] for r in template]:\n        raise ValueError(\'User identifiers/order differ from real template\')\n    if len({r[0] for r in rows}) != len(rows):\n        raise ValueError(\'Duplicate users\')\n    allowed=set(map(int,candidates))-{0}\n    for user,ranking in rows:\n        ids=[int(x) for x in ranking.split()]\n        if len(ids)!=5 or len(set(ids))!=5 or not set(ids)<=allowed:\n            raise ValueError(f\'Invalid ranking for user {user}\')\n    return {\'users\':len(rows),\'header\':header,\'five_unique_valid_ids\':True}\n', encoding="utf-8")


In [ ]:
Path('als.py').write_text('"""Weighted implicit ALS (Hu, Koren & Volinsky), exact normal-equation solver.\n\np_ui=1 if observed, else 0. c_ui=1+alpha*log1p(n_ui) if observed,\nelse 1. Minimize sum c_ui*(p_ui-x_u@y_i)^2 + reg*(||X||²+||Y||²).\nNo unobserved pair is materialized. The missing-data baseline is Y.T@Y.\n"""\nimport numpy as np\nfrom numba import njit,prange,set_num_threads\nfrom threadpoolctl import threadpool_limits\n\n@njit(parallel=True,cache=False)\ndef solve_side(indptr,indices,confidence,fixed,regularization):\n    factors=fixed.shape[1]\n    gram=fixed.T@fixed\n    result=np.zeros((len(indptr)-1,factors),np.float64)\n    for row in prange(len(indptr)-1):\n        lo,hi=indptr[row],indptr[row+1]\n        if lo==hi:\n            continue\n        a=gram.copy()\n        b=np.zeros(factors,np.float64)\n        for j in range(factors): a[j,j]+=regularization\n        for ptr in range(lo,hi):\n            idx=indices[ptr];c=confidence[ptr]\n            for j in range(factors):\n                b[j]+=c*fixed[idx,j]\n                for k in range(factors):\n                    a[j,k]+=(c-1)*fixed[idx,j]*fixed[idx,k]\n        result[row]=np.linalg.solve(a,b)\n    return result\n\nclass ImplicitALS:\n    def __init__(self,factors=32,regularization=.1,iterations=20,random_state=2026,num_threads=2):\n        self.factors=factors;self.regularization=regularization\n        self.iterations=iterations;self.random_state=random_state;self.num_threads=num_threads\n\n    def fit(self,confidence):\n        set_num_threads(self.num_threads)\n        csr=confidence.tocsr().astype(np.float64);csc=csr.T.tocsr()\n        rng=np.random.default_rng(self.random_state)\n        self.item_factors=rng.normal(0,.01,(csr.shape[1],self.factors))\n        with threadpool_limits(limits=1):\n            for iteration in range(self.iterations):\n                self.user_factors=solve_side(csr.indptr,csr.indices,csr.data,self.item_factors,self.regularization)\n                self.item_factors=solve_side(csc.indptr,csc.indices,csc.data,self.user_factors,self.regularization)\n        return self\n\ndef verify_solver():\n    """Independent dense objective verifies one update and cold-user behavior."""\n    from scipy.sparse import csr_matrix\n    fixed=np.array([[.1,.2],[.3,-.2],[.5,.4]],dtype=np.float64)\n    conf=csr_matrix([[3.,0.,5.],[0.,0.,0.]])\n    actual=solve_side(conf.indptr,conf.indices,conf.data,fixed,.1)\n    c=np.array([3.,1.,5.]);p=np.array([1.,0.,1.])\n    expected=np.linalg.solve(fixed.T@np.diag(c)@fixed+.1*np.eye(2),fixed.T@(c*p))\n    assert np.allclose(actual[0],expected,atol=1e-10)\n    assert np.allclose(actual[1],0)\n    return {\'dense_normal_equation_max_error\':float(np.max(np.abs(actual[0]-expected))),\'cold_user_zero_factors\':True}\n', encoding="utf-8")


In [ ]:
Path('models.py').write_text('"""Models fit ONLY on the caller\'s training-window interactions and metadata.\n\nPrediction candidates contain IDs only. Future schedule metadata is never used\nto fit content features. Repeats are allowed consistently across all methods.\n"""\nimport time\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.preprocessing import normalize\nfrom als import ImplicitALS\n\nSEED=2026\nATTRIBUTES=[\'tv_show_category\',\'tv_show_genre_1\',\'tv_show_genre_2\',\n            \'tv_show_genre_3\',\'year_of_production\',\'director\',\'actors\']\n\ndef content_tokens(row):\n    """Field-qualified attribute tokens; no item ID or future data feature."""\n    tokens=[]\n    for field in ATTRIBUTES:\n        value=row.get(field)\n        if pd.isna(value) or str(value).strip() in (\'\',\'0\',\'nan\',\'None\'):\n            continue\n        # Preserve full actor/director names; split common list delimiters only.\n        values=str(value).lower().replace(\';\',\',\').split(\',\')\n        for value in values:\n            value=\'_\'.join(value.strip().split())\n            if value:\n                tokens.append(field+\'=\'+value)\n    return tokens\n\nclass TrainingBundle:\n    def __init__(self,interactions,metadata,users):\n        # interactions represent qualified distinct airings, aggregated by user-item.\n        self.users=list(users)\n        self.uindex={u:i for i,u in enumerate(self.users)}\n        self.items=sorted(set(interactions.tv_show_id)-{0})\n        if not self.items:\n            raise ValueError(\'No training interactions\')\n        self.iindex={x:i for i,x in enumerate(self.items)}\n        counts=interactions.groupby([\'user_id\',\'tv_show_id\']).size().rename(\'count\').reset_index()\n        counts=counts[counts.user_id.isin(self.uindex)]\n        self.counts=csr_matrix((counts[\'count\'].astype(np.float32),\n             (counts.user_id.map(self.uindex),counts.tv_show_id.map(self.iindex))),\n             shape=(len(self.users),len(self.items)))\n        self.weights=self.counts.copy()\n        self.weights.data=np.log1p(self.weights.data)\n        self.popularity=np.asarray((self.counts>0).sum(axis=0)).ravel()\n        self.metadata=metadata.drop_duplicates(\'tv_show_id\').set_index(\'tv_show_id\')\n        self.cold=np.asarray(self.counts.sum(axis=1)).ravel()==0\n\n    def candidate_map(self,candidates):\n        c=np.array(sorted(set(map(int,candidates))-{0}),dtype=np.int64)\n        if len(c)<5:\n            raise ValueError(\'Fewer than five candidates\')\n        ci=np.array([self.iindex.get(int(x),-1) for x in c])\n        pop=np.array([self.popularity[i] if i>=0 else 0 for i in ci])\n        return c,ci,pop\n\n    def rank(self,scores,candidates):\n        """Positive scores first; fill missing/zero/negative slots with popularity.\n\n        Exact ties use training popularity then integer ID. Identical fallback\n        for every model; unseen items have popularity=0 then ID tie-break.\n        """\n        c,_,pop=self.candidate_map(candidates)\n        if scores.shape != (len(self.users),len(c)):\n            raise ValueError(\'Unexpected score matrix shape\')\n        fallback=np.lexsort((c,-pop))\n        rankings={}\n        for ui,user in enumerate(self.users):\n            s=scores[ui]\n            if self.cold[ui]:\n                indices=fallback[:5]\n            else:\n                order=np.lexsort((c,-pop,-np.nan_to_num(s,nan=-np.inf)))\n                chosen=order[np.isfinite(s[order]) & (s[order]>0)][:5].tolist()\n                for j in fallback:\n                    if len(chosen)==5:\n                        break\n                    if int(j) not in chosen:\n                        chosen.append(int(j))\n                indices=chosen\n            rankings[user]=[int(c[i]) for i in indices]\n        return rankings\n\n    def popularity_scores(self,candidates):\n        c,ci,pop=self.candidate_map(candidates)\n        return np.broadcast_to(pop,(len(self.users),len(c))).copy()\n\n    def content_scores(self,candidates):\n        # Vocabulary/IDF fit on train-only item attributes, never hold-out metadata.\n        docs=[content_tokens(row) for _,row in self.metadata.iterrows()]\n        vectorizer=TfidfVectorizer(analyzer=lambda x:x,norm=\'l2\',dtype=np.float32)\n        try:\n            vectorizer.fit(docs)\n            features=vectorizer.transform([content_tokens(self.metadata.loc[x]) if x in self.metadata.index else [] for x in self.items])\n        except ValueError as error:\n            if \'empty vocabulary\' not in str(error):\n                raise\n            return np.zeros((len(self.users),len(self.candidate_map(candidates)[0])),np.float32)\n        profiles=normalize(self.weights@features)\n        c,ci,_=self.candidate_map(candidates)\n        scores=np.zeros((len(self.users),len(c)),np.float32)\n        candidate_features=vectorizer.transform([content_tokens(self.metadata.loc[int(x)]) if int(x) in self.metadata.index else [] for x in c])\n        scores=(profiles@candidate_features.T).toarray()\n        self.content_vectorizer=vectorizer\n        return scores\n\n    def als_scores(self,candidates,factors=32,regularization=.1,alpha=20,iterations=20):\n        # implicit expects OBSERVED confidence values, missing entries mean c=1.\n        confidence=self.weights.copy()\n        confidence.data=1+alpha*confidence.data\n        model=ImplicitALS(factors=factors,regularization=regularization,\n             iterations=iterations,random_state=SEED,num_threads=2)\n        model.fit(confidence)\n        c,ci,_=self.candidate_map(candidates)\n        scores=np.zeros((len(self.users),len(c)),np.float32)\n        known=ci>=0\n        scores[:,known]=model.user_factors@model.item_factors[ci[known]].T\n        self.als_model=model\n        return scores\n\ndef reciprocal_rank_fusion(rankings_a,rankings_b,weight=.5,offset=60):\n    """Fuse full candidate rankings supplied by caller; evaluate hybrid separately."""\n    if set(rankings_a)!=set(rankings_b):\n        raise ValueError(\'Cohort mismatch\')\n    result={}\n    for user in rankings_a:\n        score={}\n        for ranking,w in [(rankings_a[user],weight),(rankings_b[user],1-weight)]:\n            for rank,item in enumerate(ranking,1):\n                score[item]=score.get(item,0)+w/(offset+rank)\n        result[user]=sorted(score,key=lambda x:(-score[x],x))[:5]\n    return result\n', encoding="utf-8")


In [ ]:
Path('test_core.py').write_text('"""Synthetic contract checks; these are AGENT checks, not student verification."""\nimport random\nimport unittest\nfrom core import ScheduleIndex,aggregate_watches,ap_at_k,map_at_k,union_length\n\nclass CoreTests(unittest.TestCase):\n    def test_ap_duplicate_empty_and_cutoff(self):\n        self.assertAlmostEqual(ap_at_k([2,4],[2,2,9,4,5]),.75)\n        self.assertEqual(ap_at_k([],[1,2,3,4,5]),0)\n        self.assertEqual(ap_at_k([6],[1,2,3,4,5,6]),0)\n        self.assertEqual(ap_at_k(range(1,11),list(range(1,6))),1)\n\n    def test_fixed_cohort_includes_empty_truth(self):\n        self.assertEqual(map_at_k({1:[2]},{1:[2],2:[3]},[1,2]),.5)\n\n    def test_interval_boundaries_channels_unlabelled(self):\n        idx=ScheduleIndex([(1,0,100,10,\'a\'),(1,100,200,20,\'b\'),\n                           (1,0,300,0,\'unlabelled\'),(2,0,200,30,\'c\')])\n        self.assertEqual(idx.overlaps(1,50,150),[(10,\'a\',50,100,100),(20,\'b\',100,150,100)])\n        self.assertEqual(idx.overlaps(1,200,250),[])\n        self.assertEqual(idx.overlaps(9,0,100),[])\n\n    def test_union_and_temporal_clipping(self):\n        idx=ScheduleIndex([(1,0,200,10,\'a\')])\n        views=[(1,1,20,130),(1,1,40,150)]\n        train=aggregate_watches(idx,views,0,100,60)\n        self.assertEqual(train[0][\'watch_seconds\'],80)\n        self.assertEqual(aggregate_watches(idx,views,100,200,60),[])\n        self.assertEqual(union_length([(1,3),(2,4),(4,5),(10,12)]),6)\n\n    def test_index_matches_brute_force_with_overlapping_schedule(self):\n        rng=random.Random(2026)\n        rows=[]\n        for i in range(300):\n            s=rng.randrange(1000)\n            rows.append((rng.randrange(3),s,s+rng.randrange(1,500),i+1,str(i)))\n        idx=ScheduleIndex(rows)\n        for _ in range(300):\n            c,s=rng.randrange(4),rng.randrange(1500)\n            e=s+rng.randrange(1,400)\n            expected=sorted((item,key,max(s,a),min(e,b),b-a)\n                            for ch,a,b,item,key in rows if ch==c and max(s,a)<min(e,b))\n            self.assertEqual(sorted(idx.overlaps(c,s,e)),expected)\n\nif __name__==\'__main__\':\n    unittest.main()\n', encoding="utf-8")


In [ ]:
Path('revised/data.py').write_text('"""Reconstruct all positive overlaps from raw inputs, with no 300-second filter."""\nimport os,json,time,sys,hashlib\nfrom pathlib import Path\nfrom collections import defaultdict\nimport numpy as np\nimport pandas as pd\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom core import ScheduleIndex,union_length\n\nCLOCK=int(pd.Timestamp(\'2020-03-29 04:00:00\').timestamp())\ndef stamp(s):\n v=int(pd.Timestamp(s).timestamp());return v-3600 if v>=CLOCK else v\ndef seconds(s,fmt):\n a=pd.to_datetime(s,format=fmt,errors=\'raise\').astype(\'int64\')//10**9\n return a-(a>=CLOCK).astype(\'int64\')*3600\ndef save(path,obj):Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)))\ndef read_schedule(folder):\n frames=[]\n for name in [\'export_arh_11-20-final.csv\',\'export_arh_21-30-final.csv\',\'export_arh_31-42-final.csv\']:\n  p=folder/name\n  if not p.exists():p=folder/(name+\'.zip\')\n  f=pd.read_csv(p,low_memory=False);f[\'start_s\']=seconds(f.start_time,\'%d.%m.%Y %H:%M:%S\');f[\'end_s\']=f.start_s+f.duration\n  f=f[f.tv_show_id.ne(0)&f.duration.gt(0)].copy();f[\'source\']=name;frames.append(f)\n return pd.concat(frames,ignore_index=True).drop_duplicates([\'channel_id\',\'start_s\',\'end_s\',\'tv_show_id\']).sort_values(\'start_s\').reset_index(drop=True)\ndef prepare(folder,out):\n t=time.perf_counter();folder=Path(folder);out=Path(out);out.mkdir(parents=True,exist_ok=True)\n schedule=read_schedule(folder);schedule.to_pickle(out/\'schedule.pkl\')\n end=stamp(\'2020-07-27\');history=schedule[schedule.start_s.lt(end)].reset_index(drop=True);history[\'airing_key\']=np.arange(len(history))\n index=ScheduleIndex(history[[\'channel_id\',\'start_s\',\'end_s\',\'tv_show_id\',\'airing_key\']].itertuples(index=False,name=None))\n segments=defaultdict(list);audit={\'view_rows\':0,\'duration_mismatch\':0,\'positive_overlap_rows\':0,\'labelled_overlap_seconds\':0,\'total_view_seconds\':0}\n input=folder/\'dataset11-30.csv\'\n if not input.exists():input=folder/\'dataset11-30.csv.zip\'\n for f in pd.read_csv(input,dtype={\'user_id\':str},chunksize=250000):\n  starts=seconds(f.start_time,\'%Y-%m-%d %H:%M:%S\').to_numpy();ends=seconds(f.stop_time,\'%Y-%m-%d %H:%M:%S\').to_numpy()\n  audit[\'view_rows\']+=len(f);audit[\'duration_mismatch\']+=int(np.sum(ends-starts!=f.duraton.to_numpy()))\n  for u,c,a,b in zip(f.user_id,f.vsetv_id,starts,ends):\n   a=int(a);b=int(b);assert b>a\n   audit[\'total_view_seconds\']+=b-a\n   matches=index.overlaps(int(c),a,min(b,end))\n   audit[\'positive_overlap_rows\']+=bool(matches)\n   audit[\'labelled_overlap_seconds\']+=union_length([(m[2],m[3]) for m in matches])\n   for item,key,left,right,length in matches:segments[(u,key)].append((left,right,a,b))\n  print(\'RAW JOIN\',audit[\'view_rows\'],flush=True)\n assert audit[\'duration_mismatch\']==0\n records=[];duplicate_segments=0;examples=[]\n for (u,key),values in segments.items():\n  unique=set(values);duplicate_segments+=len(values)-len(unique)\n  row=history.iloc[key];length=int(row.duration)\n  ints=[(x[0],x[1]) for x in unique];watched=union_length(ints);max_single=max(b-a for a,b in ints)\n  session_count=sum(5*(b-a)>=4*length for a,b in ints)\n  records.append((u,int(row.tv_show_id),key,watched,length,max_single,session_count,len(unique)))\n  if len(examples)<10 and watched<300 and 5*watched>=4*length:\n   examples.append({\'user_id\':u,\'tv_show_id\':int(row.tv_show_id),\'channel_id\':int(row.channel_id),\'airing_start\':str(row.start_time),\'duration\':length,\'union_seconds\':watched,\'intervals\':sorted(unique)})\n interactions=pd.DataFrame(records,columns=[\'user_id\',\'tv_show_id\',\'airing_key\',\'watch_seconds\',\'programme_seconds\',\'max_single_seconds\',\'session_count80\',\'unique_sessions\'])\n assert interactions.watch_seconds.le(interactions.programme_seconds).all()\n interactions.to_pickle(out/\'all_positive.pkl\');history.to_pickle(out/\'history.pkl\')\n qualified=interactions[5*interactions.watch_seconds>=4*interactions.programme_seconds]\n audit.update(positive_user_airings=len(interactions),qualified80_user_airings=len(qualified),qualified80_under300=int(qualified.watch_seconds.lt(300).sum()),deduplicated_identical_sessions=duplicate_segments,\n              union_vs_single_different=int(((5*interactions.watch_seconds>=4*interactions.programme_seconds)&(5*interactions.max_single_seconds<4*interactions.programme_seconds)).sum()),\n              join_seconds=time.perf_counter()-t,cache_source=\'fresh raw CSV join; no previous interaction cache read\',threshold=\'5*union_seconds >= 4*programme_seconds\')\n save(out/\'data_audit.json\',audit);save(out/\'private_short_examples.json\',examples)\n print(\'PREPARED\',json.dumps(audit),flush=True)\nif __name__==\'__main__\':prepare(os.environ.get(\'SWEETTV_DATA\',\'work/data\'),os.environ.get(\'SWEETTV_REVISION_OUT\',\'work/revision3\'))\n', encoding="utf-8")


In [ ]:
Path('revised/models.py').write_text('"""Training-only count models, biased MF and personalized PageRank."""\nimport sys,time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix\nfrom sklearn.preprocessing import normalize\nfrom numba import njit,prange,set_num_threads\nfrom threadpoolctl import threadpool_limits\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom als import ImplicitALS\nfrom models import TrainingBundle\n\n@njit(parallel=True)\ndef target_update(indptr,indices,confidence,targets,fixed,bias,reg):\n gram=fixed.T@fixed;offset=-(fixed.T@bias);out=np.zeros((len(indptr)-1,fixed.shape[1]))\n for row in prange(len(indptr)-1):\n  a=gram.copy();b=offset.copy()\n  for j in range(fixed.shape[1]):a[j,j]+=reg[j]\n  for q in range(indptr[row],indptr[row+1]):\n   i=indices[q];c=confidence[q];y=fixed[i]\n   b+=(c*targets[q]-(c-1)*bias[i])*y\n   for j in range(fixed.shape[1]):\n    for k in range(fixed.shape[1]):a[j,k]+=(c-1)*y[j]*y[k]\n  out[row]=np.linalg.solve(a,b)\n return out\n\nclass CountMF:\n def __init__(self,factors=32,regularization=.1,alpha=20,bias=False,iterations=20,bias_regularization=1.):\n  self.factors=factors;self.reg=regularization;self.alpha=alpha;self.use_bias=bias;self.iterations=iterations;self.bias_reg=bias_regularization\n def fit(self,counts):\n  set_num_threads(2);c=counts.astype(np.float64).tocsr();ct=c.T.tocsr()\n  target=np.log1p(c.data);target_t=np.log1p(ct.data)\n  conf=1+self.alpha*target;conf_t=1+self.alpha*target_t\n  y=np.random.default_rng(2026).normal(0,.01,(c.shape[1],self.factors));b=np.zeros(c.shape[1])\n  with threadpool_limits(limits=1):\n   for iteration in range(self.iterations):\n    x=target_update(c.indptr,c.indices,conf,target,y,b,np.full(self.factors,self.reg))\n    if self.use_bias:\n     aug=np.column_stack([x,np.ones(c.shape[0])]);reg=np.r_[np.full(self.factors,self.reg),self.bias_reg]\n     result=target_update(ct.indptr,ct.indices,conf_t,target_t,aug,np.zeros(c.shape[0]),reg)\n     y=result[:,:-1].copy();b=result[:,-1].copy()\n    else:y=target_update(ct.indptr,ct.indices,conf_t,target_t,x,np.zeros(c.shape[0]),np.full(self.factors,self.reg))\n  self.user_factors=x;self.item_factors=y;self.item_bias=b;return self\n def predict(self,rows):return self.user_factors[rows]@self.item_factors.T+self.item_bias\n\ndef verify_target_update():\n fixed=np.array([[.1,.2],[.3,-.2],[.5,.4]]);bias=np.array([.2,-.1,.3]);reg=np.array([.7,.8])\n c=csr_matrix([[3.,0.,5.],[0.,0.,0.]]);targets=np.log1p(np.array([2.,4.]));actual=target_update(c.indptr,c.indices,c.data,targets,fixed,bias,reg)\n W=np.diag([3.,1.,5.]);t=np.array([targets[0],0,targets[1]])\n expected=np.linalg.solve(fixed.T@W@fixed+np.diag(reg),fixed.T@W@(t-bias))\n cold=np.linalg.solve(fixed.T@fixed+np.diag(reg),-fixed.T@bias)\n assert np.allclose(actual[0],expected,atol=1e-10) and np.allclose(actual[1],cold,atol=1e-10)\n return float(np.max(np.abs(actual[0]-expected)))\n\nclass Window:\n def __init__(self,train,schedule,cohort,candidates,start,cutoff,end):\n  self.train=train;self.schedule=schedule;self.cohort=cohort;self.start=start;self.cutoff=cutoff;self.end=end\n  self.users=sorted(set(train.user_id)|set(cohort));self.uindex={u:i for i,u in enumerate(self.users)}\n  self.items=np.array(sorted(train.tv_show_id.unique()),dtype=int);self.iindex={i:j for j,i in enumerate(self.items)}\n  self.candidates=np.array(sorted(candidates),dtype=int);self.ci=np.array([self.iindex.get(i,-1) for i in self.candidates]);self.known=self.ci>=0\n  pairs=train.groupby([\'user_id\',\'tv_show_id\']).size().rename(\'n\').reset_index()\n  self.counts=csr_matrix((pairs.n.to_numpy(float),(pairs.user_id.map(self.uindex),pairs.tv_show_id.map(self.iindex))),shape=(len(self.users),len(self.items)))\n  self.rows=np.array([self.uindex[u] for u in cohort]);self.cold=np.asarray(self.counts[self.rows].sum(axis=1)).ravel()==0\n  self.pop=np.asarray(self.counts.sum(axis=0)).ravel();self.cpop=np.zeros(len(self.candidates));self.cpop[self.known]=self.pop[self.ci[self.known]]\n  self.repeat=self.project(self.counts[self.rows].toarray())\n def project(self,s):\n  out=np.zeros((len(self.cohort),len(self.candidates)));out[:,self.known]=s[:,self.ci[self.known]];return out\n def rank(self,scores):\n  fallback=np.lexsort((self.candidates,-self.cpop));out={}\n  for u,row,cold in zip(self.cohort,scores,self.cold):\n   if cold:chosen=fallback[:5]\n   else:\n    order=np.lexsort((self.candidates,-self.cpop,-np.nan_to_num(row,nan=-np.inf)))\n    chosen=order[np.isfinite(row[order])&(row[order]>0)][:5].tolist()\n    for j in fallback:\n     if len(chosen)==5:break\n     if int(j) not in chosen:chosen.append(int(j))\n   out[u]=self.candidates[chosen].tolist()\n  return out\n def individual(self,config):\n  family=config[\'family\'];p=config.get(\'params\',{})\n  if family==\'popularity\':return np.broadcast_to(self.cpop,(len(self.cohort),len(self.candidates))).copy()\n  if family==\'repeat\':return self.repeat.copy()\n  if family==\'content\':\n   metadata=self.schedule[(self.schedule.start_s>=self.start)&(self.schedule.end_s<=self.cutoff)]\n   bundle=TrainingBundle(self.train,metadata,self.users)\n   return bundle.content_scores(self.candidates)[self.rows]\n  if family==\'implicit\':\n   c=self.counts.copy();c.data=1+20*np.log1p(c.data)\n   model=ImplicitALS(factors=p.get(\'factors\',32),regularization=p.get(\'regularization\',.1),iterations=20,random_state=2026,num_threads=2).fit(c)\n   return self.project(model.user_factors[self.rows]@model.item_factors.T)\n  if family==\'count_mf\':return self.project(CountMF(**p).fit(self.counts).predict(self.rows))\n  if family==\'ppr\':\n   damping=p.get(\'damping\',.85);neighbors=p.get(\'neighbors\',50)\n   binary=(self.counts>0).astype(float);co=(binary.T@binary).toarray();norm=np.sqrt(np.diag(co));co/=np.maximum(norm[:,None]*norm[None,:],1e-12);np.fill_diagonal(co,0)\n   rows=[];cols=[];vals=[]\n   for i,row in enumerate(co):\n    ids=np.lexsort((self.items,-row))[:neighbors];ids=ids[row[ids]>0]\n    rows.extend([i]*len(ids));cols.extend(ids);vals.extend(row[ids])\n   P=normalize(csr_matrix((vals,(rows,cols)),shape=co.shape),norm=\'l1\',axis=1)\n   restart=normalize(self.counts[self.rows],norm=\'l1\',axis=1).toarray();score=restart.copy()\n   for iteration in range(100):\n    propagated=np.asarray(score@P);lost=score.sum(axis=1)-propagated.sum(axis=1)\n    new=(1-damping)*restart+damping*(propagated+lost[:,None]*restart)\n    delta=np.max(np.abs(new-score));score=new\n    if delta<1e-10:break\n   self.ppr_iterations=iteration+1\n   return self.project(score)\n  raise ValueError(family)\n def schedule_ratio(self):\n  # Channel preferences use past qualified airings only; no future viewer signal.\n  channels=np.array(sorted(self.schedule.channel_id.unique()));chidx={c:i for i,c in enumerate(channels)}\n  x=self.train.groupby([\'user_id\',\'channel_id\']).size().rename(\'n\').reset_index()\n  weights=csr_matrix((x.n.to_numpy(float),(x.user_id.map(self.uindex),x.channel_id.map(chidx))),shape=(len(self.users),len(channels)))\n  weights=normalize(weights[self.rows],norm=\'l1\',axis=1)\n  ci={x:i for i,x in enumerate(self.candidates)}\n  def availability(a,b):\n   f=self.schedule[(self.schedule.start_s>=a)&(self.schedule.end_s<=b)&self.schedule.tv_show_id.isin(ci)]\n   z=f.groupby([\'channel_id\',\'tv_show_id\']).size().rename(\'n\').reset_index()\n   return csr_matrix((z.n.to_numpy(float)/((b-a)/86400),(z.channel_id.map(chidx),z.tv_show_id.map(ci))),shape=(len(channels),len(ci)))\n  past=(weights@availability(self.start,self.cutoff)).toarray();future=(weights@availability(self.cutoff,self.end)).toarray()\n  return np.clip((future+.02)/(past+.02),.25,4.)\n\ndef rrf(window,a,b,weight):\n def reciprocal(s):\n  out=np.empty_like(s)\n  for i,row in enumerate(s):\n   # Same score/fallback ordering before rank fusion.\n   order=np.lexsort((window.candidates,-window.cpop,-np.maximum(row,0)))\n   if window.cold[i]:order=np.lexsort((window.candidates,-window.cpop))\n   out[i,order]=1/(20+np.arange(1,len(order)+1))\n  return out\n return (1-weight)*reciprocal(a)+weight*reciprocal(b)\n', encoding="utf-8")


In [ ]:
Path('revised/experiments.py').write_text('"""Select on window A, freeze, then evaluate forward window B exactly once."""\nimport os,sys,time,json,hashlib,platform,argparse,importlib.metadata\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom threadpoolctl import threadpool_limits\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom revised.data import stamp,save\nfrom revised.models import Window,rrf,verify_target_update\nfrom als import verify_solver\nfrom core import ap_at_k,validate_submission\n\nSTART=stamp(\'2020-03-09\');A=stamp(\'2020-05-18\');B=stamp(\'2020-06-22\');END=stamp(\'2020-07-27\')\n\ndef counted(interactions,policy=\'union80\'):\n if policy==\'union80\':return interactions[5*interactions.watch_seconds>=4*interactions.programme_seconds].copy()\n if policy==\'single80\':return interactions[5*interactions.max_single_seconds>=4*interactions.programme_seconds].copy()\n if policy==\'sessions80\':return interactions.loc[interactions.index.repeat(interactions.session_count80)].copy()\n if policy==\'seconds300\':return interactions[interactions.watch_seconds>=300].copy()\n raise ValueError(policy)\ndef truth_for(events,users,ties=\'id_asc\'):\n c=events.groupby([\'user_id\',\'tv_show_id\']).size().rename(\'count\').reset_index();out={};frequencies={};ambiguous=0\n for u,f in c.groupby(\'user_id\'):\n  if ties==\'hash\':\n   f=f.copy();f[\'tie\']=[int(hashlib.sha256(f\'2026:{x}\'.encode()).hexdigest()[:12],16) for x in f.tv_show_id]\n  else:f=f.assign(tie=f.tv_show_id if ties==\'id_asc\' else -f.tv_show_id)\n  f=f.sort_values([\'count\',\'tie\'],ascending=[False,True]);out[u]=f.tv_show_id.head(5).astype(int).tolist()\n  frequencies[u]=dict(zip(f.tv_show_id.astype(int),f[\'count\'].astype(int)))\n  if len(f)>5 and f.iloc[4][\'count\']==f.iloc[5][\'count\']:ambiguous+=int(u in users)\n return {u:out.get(u,[]) for u in users},frequencies,ambiguous\n\ndef evaluate(truth,pred,users):\n vals=[]\n for u in users:\n  actual=set(truth[u]);hit=np.array([x in actual for x in pred[u]],float)\n  ap=float(np.sum(np.cumsum(hit)/np.arange(1,6)*hit)/min(5,len(actual))) if actual else 0.\n  assert abs(ap-ap_at_k(actual,pred[u]))<1e-12;vals.append(ap)\n return float(np.mean(vals))\n\ndef load_window(out,which):\n f=pd.read_pickle(out/\'all_positive.pkl\');h=pd.read_pickle(out/\'history.pkl\');s=pd.read_pickle(out/\'schedule.pkl\')\n f=f.merge(h[[\'airing_key\',\'start_s\',\'end_s\',\'channel_id\']],on=\'airing_key\',validate=\'many_to_one\')\n template=pd.read_csv(Path(os.environ.get(\'SWEETTV_DATA\',\'work/data\'))/\'submission.csv\',dtype=str);users=template.user_id.tolist()\n cutoff,end=(A,B) if which==\'main\' else (B,END) if which==\'forward\' else (END,int(s.end_s.max()))\n train_all=f[(f.start_s>=START)&(f.end_s<=cutoff)];future_all=f[(f.start_s>=cutoff)&(f.end_s<=end)]\n train=counted(train_all);future=counted(future_all)\n truth,counts,ties=truth_for(future,set(users));eligible=[u for u in users if len(truth[u])==5]\n candidates=set(s.loc[s.start_s.lt(end)&s.end_s.gt(cutoff),\'tv_show_id\'])\n audit={\'window\':which,\'cutoff\':cutoff,\'end\':end,\'users\':len(users),\'strict5_users\':len(eligible),\'less5_users\':sum(0<len(truth[u])<5 for u in users),\'empty_users\':sum(not truth[u] for u in users),\'ties_across_fifth_users\':ties,\n        \'train_airings\':len(train),\'validation_airings\':len(future),\'train_short_qualified\':int(train.watch_seconds.lt(300).sum()),\'validation_short_qualified\':int(future.watch_seconds.lt(300).sum()),\n        \'excluded_boundary_user_airings\':int(((f.start_s<cutoff)&(f.end_s>cutoff)).sum()),\'candidates\':len(candidates),\'train_users\':int(train.user_id.nunique())}\n window=Window(train,s,users,candidates,START,cutoff,end)\n return window,truth,eligible,future_all,audit,template,counts\n\ndef configurations():\n c=[{\'id\':\'popularity\',\'family\':\'popularity\'},{\'id\':\'repeat\',\'family\':\'repeat\'},{\'id\':\'content\',\'family\':\'content\'},\n {\'id\':\'implicit32\',\'family\':\'implicit\',\'params\':{\'factors\':32,\'regularization\':.1}},\n {\'id\':\'implicit32_reg1\',\'family\':\'implicit\',\'params\':{\'factors\':32,\'regularization\':1.}},\n {\'id\':\'implicit64\',\'family\':\'implicit\',\'params\':{\'factors\':64,\'regularization\':.1}}]\n # Paired changes: target -> bias -> lambda; fixed dim/alpha until explicit variants.\n for name,params in [\n  (\'count32\',{\'factors\':32,\'regularization\':.1,\'alpha\':20,\'bias\':False}),\n  (\'count32_bias\',{\'factors\':32,\'regularization\':.1,\'alpha\':20,\'bias\':True}),\n  (\'count32_bias_reg1\',{\'factors\':32,\'regularization\':1.,\'alpha\':20,\'bias\':True}),\n  (\'count32_bias_reg10\',{\'factors\':32,\'regularization\':10.,\'alpha\':20,\'bias\':True}),\n  (\'count64_bias_reg1\',{\'factors\':64,\'regularization\':1.,\'alpha\':20,\'bias\':True}),\n  (\'count32_bias_unweighted\',{\'factors\':32,\'regularization\':1.,\'alpha\':0,\'bias\':True})]:c.append({\'id\':name,\'family\':\'count_mf\',\'params\':params})\n for name,params in [(\'ppr85\',{\'damping\':.85,\'neighbors\':50}),(\'ppr50\',{\'damping\':.5,\'neighbors\':50}),(\'ppr50_k100\',{\'damping\':.5,\'neighbors\':100})]:c.append({\'id\':name,\'family\':\'ppr\',\'params\':params})\n return c\n\ndef run(which,out):\n out=Path(out);t0=time.perf_counter();window,truth,eligible,future_all,audit,template,counts=load_window(out,which)\n save(out/f\'{which}_audit.json\',audit);save(out/f\'private_{which}_truth.json\',truth)\n # Compilation is excluded from all timed models, not just later ones.\n checks={\'weighted_target_dense_error\':verify_target_update(),\'implicit_solver\':verify_solver()};save(out/\'solver_checks.json\',checks)\n results=[];cache={};configs=configurations() if which==\'main\' else json.loads((out/\'frozen_selection.json\').read_text())[\'families\']\n def process(config):\n  cid=config[\'id\'];family=config[\'family\'];t=time.perf_counter();component_time=0.\n  if family==\'hybrid\':\n   base=config[\'component\'];scores=rrf(window,cache[\'repeat\'],cache[base],config[\'weight\']);component_time=next(r[\'seconds\'] for r in results if r[\'id\']==\'repeat\')+next(r[\'seconds\'] for r in results if r[\'id\']==base)\n  elif family==\'schedule\':\n   scores=cache[\'repeat\']*window.schedule_ratio()**config[\'beta\'];component_time=next(r[\'seconds\'] for r in results if r[\'id\']==\'repeat\')\n  else:scores=window.individual(config)\n  pred=window.rank(scores);elapsed=time.perf_counter()-t+component_time\n  row={**config,\'map5\':evaluate(truth,pred,eligible),\'all1260_map5\':evaluate(truth,pred,window.cohort),\'seconds\':elapsed}\n  results.append(row);cache[cid]=scores;save(out/f\'private_{which}_{cid}.json\',pred);save(out/f\'{which}_results.json\',results)\n  print(which,\'RESULT\',json.dumps(row),flush=True)\n for config in configs:process(config)\n if which==\'main\':\n  # Individual families are selected only by main-window MAP.\n  selected=[]\n  for family in [\'popularity\',\'repeat\',\'content\',\'implicit\',\'count_mf\',\'ppr\']:\n   selected.append(max([r for r in results if r[\'family\']==family],key=lambda r:r[\'map5\']))\n  for component in [x[\'id\'] for x in selected if x[\'family\'] in [\'count_mf\',\'ppr\']]+[\'popularity\']:\n   for weight in [.1,.25,.5]:process({\'id\':f\'hybrid_{component}_{weight}\',\'family\':\'hybrid\',\'component\':component,\'weight\':weight})\n  for beta in [.25,.5,1.]:process({\'id\':f\'schedule_{beta}\',\'family\':\'schedule\',\'beta\':beta})\n  selected.extend([max([r for r in results if r[\'family\']==family],key=lambda r:r[\'map5\']) for family in [\'hybrid\',\'schedule\']])\n  winner=max(results,key=lambda r:r[\'map5\'])\n  frozen={\'winner\':winner,\'families\':selected,\'selected_using\':\'main window only; forward scores not yet computed\',\'frozen_at_utc\':pd.Timestamp.now(tz=\'UTC\').isoformat()}\n  save(out/\'frozen_selection.json\',frozen);print(\'FROZEN\',json.dumps(frozen),flush=True)\n # Sensitivity uses frozen chosen-family predictions; no refit or retuning from these results.\n sensitivity=[]\n chosen=json.loads((out/\'frozen_selection.json\').read_text())[\'families\']\n for policy,tie in [(\'union80\',\'id_desc\'),(\'union80\',\'hash\'),(\'single80\',\'id_asc\'),(\'sessions80\',\'id_asc\')]:\n  alternative,_,_=truth_for(counted(future_all,policy),set(window.cohort),tie)\n  for config in chosen:\n   pred=json.loads((out/f\'private_{which}_{config["id"]}.json\').read_text())\n   sensitivity.append({\'model\':config[\'id\'],\'policy\':policy,\'tie\':tie,\'map5_fixed_main_cohort\':evaluate(alternative,pred,eligible),\'same_users\':len(eligible),\'altered_truth_users\':sum(set(truth[u])!=set(alternative[u]) for u in eligible)})\n save(out/f\'{which}_sensitivity.json\',sensitivity)\n save(out/f\'{which}_runtime.json\',{\'wall_seconds\':time.perf_counter()-t0,\'python\':sys.version,\'platform\':platform.platform(),\'cpu_count\':os.cpu_count(),\'versions\':{n:importlib.metadata.version(n) for n in [\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'numba\',\'threadpoolctl\']},\'seed\':2026,\'threads\':2})\n print(which,\'COMPLETE\',flush=True)\n\nif __name__==\'__main__\':\n parser=argparse.ArgumentParser();parser.add_argument(\'stage\',choices=[\'main\',\'forward\']);args=parser.parse_args()\n with threadpool_limits(limits=2):run(args.stage,os.environ.get(\'SWEETTV_REVISION_OUT\',\'work/revision3\'))\n', encoding="utf-8")


In [ ]:
Path('revised/final.py').write_text('"""Refit only the main-window frozen winner and validate the exact CSV."""\nimport os,json,time,hashlib\nfrom pathlib import Path\nfrom threadpoolctl import threadpool_limits\nfrom revised.data import save\nfrom revised.experiments import load_window\nfrom revised.models import rrf\nfrom core import validate_submission\n\ndef final(out):\n out=Path(out);frozen=json.loads((out/\'frozen_selection.json\').read_text())\n window,_,_,_,audit,template,_=load_window(out,\'final\')\n configs={r[\'id\']:r for r in frozen[\'families\']}\n def scores(c):\n  if c[\'family\']==\'hybrid\':return rrf(window,scores(configs[\'repeat\']),scores(configs[c[\'component\']]),c[\'weight\'])\n  if c[\'family\']==\'schedule\':return window.repeat*window.schedule_ratio()**c[\'beta\']\n  return window.individual(c)\n start=time.perf_counter();pred=window.rank(scores(frozen[\'winner\']));elapsed=time.perf_counter()-start\n template.iloc[:,1]=[\' \'.join(map(str,pred[u])) for u in template.iloc[:,0]]\n path=out/\'submission_corrected.csv\';template.to_csv(path,index=False)\n contract=validate_submission(Path(os.environ.get(\'SWEETTV_DATA\',\'work/data\'))/\'submission.csv\',path,window.candidates)\n save(out/\'final_manifest.json\',{\'selected\':frozen[\'winner\'],\'fit_score_rank_seconds\':elapsed,\'training_airings\':len(window.train),\n   \'training_users\':len(window.users),\'training_items\':len(window.items),\'candidates\':len(window.candidates),\n   \'csv_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\'contract\':contract,\'test_end_seconds\':window.end})\n print(\'FINAL\',json.dumps(json.loads((out/\'final_manifest.json\').read_text())),flush=True)\n\nif __name__==\'__main__\':\n with threadpool_limits(limits=2):final(os.environ.get(\'SWEETTV_REVISION_OUT\',\'work/revision3\'))\n', encoding="utf-8")


In [ ]:
Path('revised/test_target.py').write_text("import sys,unittest\nfrom pathlib import Path\nimport pandas as pd\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom core import union_length,ap_at_k\nfrom revised.experiments import counted,truth_for\nfrom revised.models import verify_target_update\n\nclass TargetTests(unittest.TestCase):\n def test_short_airing_and_exact_boundary(self):\n  f=pd.DataFrame({'watch_seconds':[96,95,300],'programme_seconds':[120,120,600]})\n  self.assertEqual(counted(f).index.tolist(),[0])\n def test_union_duplicate_and_split_sessions(self):\n  self.assertEqual(union_length([(0,60),(40,96),(0,60)]),96)\n def test_top_five_ties_and_fewer(self):\n  f=pd.DataFrame({'user_id':['u']*7+['short'],'tv_show_id':[1,1,2,3,4,5,6,9]})\n  truth,_,ties=truth_for(f,{'u','short','empty'})\n  self.assertEqual(truth['u'],[1,2,3,4,5]);self.assertEqual(truth['short'],[9]);self.assertEqual(truth['empty'],[]);self.assertEqual(ties,1)\n  alt,_,_=truth_for(f,{'u'},'id_desc');self.assertEqual(alt['u'],[1,6,5,4,3])\n def test_target_mf_dense_reference(self):self.assertLess(verify_target_update(),1e-10)\n def test_ap_five_relevance(self):self.assertAlmostEqual(ap_at_k([1,2,3,4,5],[1,8,2,9,3]),(1+2/3+3/5)/5)\nif __name__=='__main__':unittest.main()\n", encoding="utf-8")


In [ ]:
inputs=list(Path("/kaggle/input").rglob("dataset11-30.csv"))
if not inputs: inputs=list(Path("/kaggle/input").rglob("dataset11-30.csv.zip"))
assert len(inputs)==1
os.environ["SWEETTV_DATA"]=str(inputs[0].parent)
os.environ["SWEETTV_REVISION_OUT"]="/kaggle/working/results"
os.environ["PYTHONPATH"]="/kaggle/working"
print("Input folder:", inputs[0].parent)
subprocess.run([sys.executable,"revised/test_target.py"],check=True)


## Raw reconstruction and main-window selection
Fresh positive overlaps, unioned per user/airing; >=80% coverage, then five most frequent programme IDs. Unspecified tie/session conventions have sensitivity checks.

In [ ]:
subprocess.run([sys.executable,"-m","revised.data"],check=True)
subprocess.run([sys.executable,"-m","revised.experiments","main"],check=True)


## Forward check and full-history refit
Forward results do not change the frozen choice. The CSV is checked against the template; submission is a separate authorized step.

In [ ]:
subprocess.run([sys.executable,"-m","revised.experiments","forward"],check=True)
subprocess.run([sys.executable,"-m","revised.final"],check=True)
import json
print(json.loads(Path("results/final_manifest.json").read_text()))
# Raw caches need not be downloaded; viewing histories remain private.
for p in Path("results").glob("*.pkl"):p.unlink()
